In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip uninstall -y mamba-ssm causal-conv1d -q
!TORCH_CUDA_ARCH_LIST="7.5" pip install causal-conv1d --no-build-isolation --no-cache-dir -q
!TORCH_CUDA_ARCH_LIST="7.5" pip install mamba-ssm --no-build-isolation --no-cache-dir -q
!pip install datasets transformers -q

In [ ]:
# VMamba needs mamba_ssm + triton. The prebuilt wheel matches Kaggle's torch2.2+CUDA12 env.
!pip install -q timm==0.4.12 einops fvcore
!pip install -q causal-conv1d
!pip install -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.2cxx11abiTRUE-cp310-cp310-linux_x86_64.whl
!pip install -q pycocotools transformers
print("Done")

In [ ]:
# ── PATHS ── corrected to match your actual mounts ───────────────────────────
TEXT_CKPT_DIR  = "/kaggle/input/notebooks/johnmethew/mamba-1-text-tower"
TEXT_CKPT_PATH = "/kaggle/input/notebooks/johnmethew/mamba-1-text-tower/ckpt/best_model.pt"
COCO_IMG_TRAIN = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017"
COCO_IMG_VAL   = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017"
COCO_ANN_TRAIN = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_train2017.json"
COCO_ANN_VAL   = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/captions_val2017.json"
WORK_DIR       = "/kaggle/working"
VMAMBA_CKPT    = f"{WORK_DIR}/vssm_base_ckpt.pth"
VMAMBA_PY      = f"{WORK_DIR}/vmamba.py"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_DIM  = 512
BATCH_SIZE = 64
LR         = 5e-4
EPOCHS     = 10
MAX_SEQ_LEN = 77
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import urllib.request

VMAMBA_CKPT = f"{WORK_DIR}/vssm_base_ckpt.pth"
VMAMBA_PY   = f"{WORK_DIR}/vmamba.py"

# vmamba.py — single self-contained file (no repo clone needed)
if not os.path.exists(VMAMBA_PY):
    print("Downloading vmamba.py...")
    url = "https://raw.githubusercontent.com/MzeroMiko/VMamba/main/vmamba.py"
    urllib.request.urlretrieve(url, VMAMBA_PY)
    print("✓ vmamba.py")

# VMamba-Base ImageNet-1K pretrained checkpoint (~340MB)
if not os.path.exists(VMAMBA_CKPT):
    print("Downloading VMamba-B checkpoint (~340MB)...")
    !wget -q "https://github.com/MzeroMiko/VMamba/releases/download/%23v2cls/vssm_base_0229_ckpt_epoch_237.pth" \
         -O {VMAMBA_CKPT}
    print("✓ VMamba-B checkpoint")

print("VMamba ckpt size:", os.path.getsize(VMAMBA_CKPT) // 1e6, "MB")

In [ ]:
# Find the checkpoint file
ckpt_files = glob.glob(f"{TEXT_CKPT_DIR}/**/*.pth", recursive=True) + \
             glob.glob(f"{TEXT_CKPT_DIR}/**/*.pt", recursive=True)  + \
             glob.glob(f"{TEXT_CKPT_DIR}/**/*.ckpt", recursive=True)
print("Found checkpoints:", ckpt_files)

TEXT_CKPT_PATH = ckpt_files[0]  # adjust index if multiple
ckpt = torch.load(TEXT_CKPT_PATH, map_location="cpu")

# Print top-level keys so we know the structure
if isinstance(ckpt, dict):
    print("Keys:", list(ckpt.keys())[:10])
    # might be: {'model': ..., 'state_dict': ..., 'epoch': ..., etc.}
    state = ckpt.get("model", ckpt.get("state_dict", ckpt))
    sample_keys = list(state.keys())[:15]
    print("\nModel state_dict sample keys:")
    for k in sample_keys:
        print(f"  {k}: {state[k].shape}")

In [ ]:
# ── FAILSAFE: patch vmamba.py's key remapping to be shape-safe ─────────────
with open(VMAMBA_PY, "r") as f:
    src = f.read()

old = ('                state_dict[prefix + v] = state_dict[prefix + k]'
       '.view_as(self_state_dict[v])\n'
       '                state_dict.pop(prefix + k)')
new = ('                try:\n'
       '                    state_dict[prefix + v] = state_dict[prefix + k]'
       '.view_as(self_state_dict[v])\n'
       '                    state_dict.pop(prefix + k)\n'
       '                except RuntimeError:\n'
       '                    pass  # shape mismatch in key remap — skip safely')

if old in src:
    src = src.replace(old, new)
    with open(VMAMBA_PY, "w") as f:
        f.write(src)
    print("✓ vmamba.py patched — rerun Cell 5 now")
else:
    print("Pattern not found — vmamba.py may already be patched or has different whitespace")
    print("Search for 'view_as(self_state_dict' manually in the file")

In [ ]:
import inspect, importlib, sys

# fresh import
if "vmamba" in sys.modules:
    del sys.modules["vmamba"]
from vmamba import VSSM

# 1. See all VSSM constructor params
print("═══ VSSM constructor params ═══")
sig = inspect.signature(VSSM.__init__)
for name, p in sig.parameters.items():
    if name == "self": continue
    print(f"  {name:30s}  default={p.default}")

# 2. Print every checkpoint key so we know the full architecture
print("\n═══ VMamba checkpoint keys ═══")
ckpt  = torch.load(VMAMBA_CKPT, map_location="cpu", weights_only=False)
state = ckpt.get("model", ckpt)
print(f"Total keys: {len(state)}\n")
for k, v in state.items():
    print(f"  {k:60s} {str(tuple(v.shape)):>25}")

In [ ]:
if "vmamba" in sys.modules:
    del sys.modules["vmamba"]
from vmamba import VSSM
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

class VisionMambaEncoder(nn.Module):
    def __init__(self, ckpt_path):
        super().__init__()

        self.backbone = VSSM(
            in_chans            = 3,
            patch_size          = 4,
            num_classes         = 1000,
            dims                = [128, 256, 512, 1024],
            depths              = [2, 2, 15, 2],
            ssm_d_state         = 1,
            ssm_ratio           = 2.0,
            ssm_dt_rank         = "auto",
            ssm_act_layer       = "silu",
            ssm_conv            = 3,
            ssm_conv_bias       = False,
            ssm_drop_rate       = 0.0,
            ssm_init            = "v0",
            forward_type        = "v05_noz",
            mlp_ratio           = 4.0,
            mlp_act_layer       = "gelu",
            mlp_drop_rate       = 0.0,
            gmlp                = False,
            drop_path_rate      = 0.0,
            patch_norm          = True,
            patchembed_version  = "v2",
            downsample_version  = "v3",
        )

        ckpt  = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = ckpt.get("model", ckpt)
        missing, unexpected = self.backbone.load_state_dict(state, strict=False)
        print(f"VMamba-B loaded ✓  |  missing={len(missing)}  unexpected={len(unexpected)}")

        # ── KEY FIX ────────────────────────────────────────────────────────────
        # classifier does: spatial → avgpool → norm(1024) → head(1000)
        # Replace head with Identity so forward(x) returns (B, 1024) features
        self.backbone.classifier.head = nn.Identity()
        # ───────────────────────────────────────────────────────────────────────

        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()
        self.out_dim = 1024

    @torch.no_grad()
    def forward(self, x):
        # backbone.forward now returns (B, 1024) — pooled + normed, head removed
        feats = self.backbone(x)

        # Guard: if classifier didn't pool (returned spatial tensor)
        if feats.dim() == 4:    # (B, H, W, C)
            feats = feats.mean(dim=(1, 2))
        elif feats.dim() == 3:  # (B, L, C)
            feats = feats.mean(dim=1)
        # dim==2 → already (B, 1024), use as-is

        return feats


vis_enc = VisionMambaEncoder(VMAMBA_CKPT).to(DEVICE).half()

with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224, device=DEVICE, dtype=torch.float16)
    out   = vis_enc(dummy)

print(f"Vision encoder output : {out.shape}")          # expect (2, 1024)
print(f"Output norm           : {out.norm(dim=-1)}")
print(f"Trainable params      : {sum(p.numel() for p in vis_enc.parameters() if p.requires_grad)}")
# Trainable params should be 0

In [ ]:
# Replace nn.LayerNorm in TextMambaEncoder with RMSNorm (no bias, matches checkpoint)
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

# In TextMambaEncoder.__init__, change:
#   "norm": nn.LayerNorm(d_model)
# to:
#   "norm": RMSNorm(d_model)
# Missing keys will drop to 0.

In [ ]:
# Diagnostic — only run if norm_f is missing
final_norm_keys = [k for k in remapped if "norm" in k and "layers" not in k]
print("Final norm keys in checkpoint:", final_norm_keys)
# e.g. ['norm_f.weight', 'norm_f.bias'] or ['norm.weight', 'norm.bias']

In [ ]:
from mamba_ssm import Mamba

class TextMambaEncoder(nn.Module):
    """
    Matches the checkpoint architecture exactly:
      backbone.embedding.weight          [50280, 768]
      backbone.layers.N.norm.weight      [768]        ← pre-norm
      backbone.layers.N.mixer.*          Mamba(d_model=768, d_state=16, expand=2, d_conv=4)
      backbone.norm_f.weight             [768]        ← final norm (may exist)
    """
    def __init__(self, vocab_size=50280, d_model=768, n_layers=16, d_state=16,
                 d_conv=4, expand=2, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "norm":  RMSNorm(d_model),
                "mixer": Mamba(d_model=d_model, d_state=d_state,
                               d_conv=d_conv, expand=expand)
            }) for _ in range(n_layers)
        ])
        self.norm_f = nn.LayerNorm(d_model)   # final norm
        self.out_dim = d_model                 # 768

    def forward(self, input_ids):
        x = self.embedding(input_ids)          # (B, L, 768)
        for layer in self.layers:
            x = x + layer["mixer"](layer["norm"](x))   # pre-norm residual
        x = self.norm_f(x)
        return x.mean(dim=1)                   # mean pool → (B, 768)


text_enc = TextMambaEncoder(
    vocab_size=50280, d_model=768, n_layers=16,
    d_state=16, d_conv=4, expand=2
).to(DEVICE)

# ── Load checkpoint (strip the "backbone." prefix from keys) ─────────────────
raw = torch.load(TEXT_CKPT_PATH, map_location="cpu", weights_only=False)
state = raw["model"]   # confirmed key from Cell 4

# Build remapped state dict: "backbone.embedding.weight" → "embedding.weight", etc.
remapped = {}
for k, v in state.items():
    if k.startswith("backbone."):
        new_k = k[len("backbone."):]    # strip prefix
        remapped[new_k] = v
    # skip anything outside backbone (lm_head, etc.) — not needed for retrieval

missing, unexpected = text_enc.load_state_dict(remapped, strict=False)
print(f"Text Mamba loaded ✓")
print(f"  Missing   : {len(missing)}")
print(f"  Unexpected: {len(unexpected)}")
if missing:
    print(f"  Sample missing: {missing[:5]}")
    # if norm_f is missing it just means checkpoint uses different final norm name
    # check with: [k for k in remapped if "norm" in k and "layers" not in k]

# Freeze
for p in text_enc.parameters():
    p.requires_grad = False
text_enc.eval()
text_enc = text_enc.half()

# Sanity check
with torch.no_grad():
    dummy_ids = torch.randint(0, 50280, (2, 32), device=DEVICE)
    out = text_enc(dummy_ids)
print(f"Text encoder output: {out.shape}")   # expect (2, 768)

In [6]:
class ProjectionHead(nn.Module):
    """MLP: in_dim → hidden → out_dim with GELU + LayerNorm (same as CLIP)"""
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        return self.norm(self.net(x.float()))   # float32 for proj head

class CLIMP(nn.Module):
    def __init__(self, vis_encoder, text_encoder, embed_dim=512):
        super().__init__()
        self.vis_enc  = vis_encoder
        self.text_enc = text_encoder
        # Projection heads — these are the ONLY trained params
        self.vis_proj  = ProjectionHead(vis_encoder.out_dim,  vis_encoder.out_dim,  embed_dim)
        self.text_proj = ProjectionHead(text_encoder.out_dim, text_encoder.out_dim, embed_dim)
        self.logit_scale = nn.Parameter(torch.ones([]) * math.log(1 / TEMPERATURE))

    def encode_image(self, images):
        with torch.no_grad():
            feats = self.vis_enc(images)
        return F.normalize(self.vis_proj(feats), dim=-1)

    def encode_text(self, input_ids):
        with torch.no_grad():
            feats = self.text_enc(input_ids)
        return F.normalize(self.text_proj(feats), dim=-1)

    def forward(self, images, input_ids):
        img_emb  = self.encode_image(images)
        text_emb = self.encode_text(input_ids)
        scale    = self.logit_scale.exp().clamp(max=100)
        logits_i2t = scale * img_emb @ text_emb.T
        logits_t2i = logits_i2t.T
        return logits_i2t, logits_t2i

model = CLIMP(vis_enc, text_enc, embed_dim=EMBED_DIM).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / Total: {total:,}")
# Should see ~only projection head params trainable (~2-4M)

NameError: name 'vis_enc' is not defined

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class COCOCaptionsDataset(Dataset):
    def __init__(self, img_dir, ann_file, transform=None, max_len=MAX_SEQ_LEN):
        with open(ann_file) as f:
            data = json.load(f)
        self.img_dir  = img_dir
        self.transform = transform
        self.max_len  = max_len
        # Build id→filename map
        id2file = {img["id"]: img["file_name"] for img in data["images"]}
        # One row per caption
        self.pairs = [(id2file[ann["image_id"]], ann["caption"])
                      for ann in data["annotations"]]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        fname, caption = self.pairs[idx]
        img_path = os.path.join(self.img_dir, fname)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        tok = tokenizer(caption, max_length=self.max_len,
                        padding="max_length", truncation=True,
                        return_tensors="pt")
        return image, tok["input_ids"].squeeze(0)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = COCOCaptionsDataset(COCO_IMG_TRAIN, COCO_ANN_TRAIN, train_transform)
val_ds   = COCOCaptionsDataset(COCO_IMG_VAL,   COCO_ANN_VAL,   val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds):,} pairs | Val: {len(val_ds):,} pairs")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ── EMERGENCY SAVE — images are done, save before kernel dies ─────────────────
import numpy as np, os
CACHE_DIR = f"{WORK_DIR}/feature_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# These are still in memory from the interrupted cell
np.save(f"{CACHE_DIR}/tr_img.npy",    tr_img_feats)
np.save(f"{CACHE_DIR}/tr_imgids.npy", np.array(tr_img_ids))
print(f"✓ Saved train image features {tr_img_feats.shape}")
print(f"  File size: {os.path.getsize(CACHE_DIR+'/tr_img.npy')/1e6:.1f} MB")

In [ ]:
CACHE_DIR = f"{WORK_DIR}/feature_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
N_GPUS = torch.cuda.device_count()

# ── Dataset classes (unchanged) ───────────────────────────────────────────────
class ImageOnlyDS(Dataset):
    def __init__(self, img_dir, ann_file, tfm):
        with open(ann_file) as f: data = json.load(f)
        self.img_dir = img_dir; self.tfm = tfm
        self.imgs = [(x['id'], x['file_name']) for x in data['images']]
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        iid, fname = self.imgs[i]
        img = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        return self.tfm(img), iid

class CaptionOnlyDS(Dataset):
    def __init__(self, ann_file, tok, max_len):
        with open(ann_file) as f: data = json.load(f)
        self.anns = [(x['image_id'], x['caption']) for x in data['annotations']]
        self.tok = tok; self.max_len = max_len
    def __len__(self): return len(self.anns)
    def __getitem__(self, i):
        iid, cap = self.anns[i]
        t = self.tok(cap, max_length=self.max_len, padding='max_length',
                     truncation=True, return_tensors='pt')
        return t['input_ids'].squeeze(0), iid

encode_tfm = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def encode_images(enc, ds, bs=256):
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
    feats, ids = [], []
    enc.eval()
    with torch.no_grad():
        for imgs, iids in tqdm(dl):
            feats.append(enc(imgs.to(DEVICE).half()).cpu().float().numpy())
            ids.extend(iids.tolist())
    return np.concatenate(feats), ids

def encode_texts_chunked(enc, ds, bs=1024, chunk_batches=100, save_dir=None, prefix=""):
    """
    Encodes texts and saves every chunk_batches batches to disk.
    On resume, skips already-saved chunks automatically.
    """
    dl      = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
    total   = len(dl)
    enc.eval()

    chunk_idx  = 0
    batch_buf_f, batch_buf_id = [], []

    # Find how many chunks already done
    done_chunks = sorted([
        int(f.split("_")[-1].replace(".npy",""))
        for f in os.listdir(save_dir)
        if f.startswith(f"{prefix}_feat_chunk")
    ]) if save_dir else []
    skip_batches = len(done_chunks) * chunk_batches
    print(f"  Resuming: skipping {skip_batches}/{total} batches "
          f"({len(done_chunks)} chunks already saved)")

    with torch.no_grad():
        for batch_i, (toks, iids) in enumerate(tqdm(dl, total=total)):
            if batch_i < skip_batches:
                continue   # skip already done batches

            out = enc(toks.to(DEVICE))
            batch_buf_f.append(out.cpu().float().numpy())
            batch_buf_id.extend(iids.tolist())

            # Save chunk every chunk_batches batches or at end
            is_last  = (batch_i == total - 1)
            is_chunk = ((batch_i - skip_batches + 1) % chunk_batches == 0)
            if (is_chunk or is_last) and batch_buf_f:
                cid = len(done_chunks)
                np.save(f"{save_dir}/{prefix}_feat_chunk_{cid}.npy",
                        np.concatenate(batch_buf_f))
                np.save(f"{save_dir}/{prefix}_ids_chunk_{cid}.npy",
                        np.array(batch_buf_id))
                done_chunks.append(cid)
                batch_buf_f, batch_buf_id = [], []
                print(f"  ✓ Chunk {cid} saved  ({batch_i+1}/{total} batches done)")

    # Concatenate all chunks
    all_feats = np.concatenate([
        np.load(f"{save_dir}/{prefix}_feat_chunk_{i}.npy") for i in range(len(done_chunks))
    ])
    all_ids = np.concatenate([
        np.load(f"{save_dir}/{prefix}_ids_chunk_{i}.npy")  for i in range(len(done_chunks))
    ])
    return all_feats, all_ids.tolist()

# ── TRAIN IMAGES ──────────────────────────────────────────────────────────────
if os.path.exists(f"{CACHE_DIR}/tr_img.npy"):
    print("✓ Train images: loading from cache")
    tr_img_feats = np.load(f"{CACHE_DIR}/tr_img.npy")
    tr_img_ids   = np.load(f"{CACHE_DIR}/tr_imgids.npy").tolist()
else:
    print("─── Encoding train images ───")
    tr_img_feats, tr_img_ids = encode_images(
        vis_enc, ImageOnlyDS(COCO_IMG_TRAIN, COCO_ANN_TRAIN, encode_tfm))
    np.save(f"{CACHE_DIR}/tr_img.npy",    tr_img_feats)
    np.save(f"{CACHE_DIR}/tr_imgids.npy", np.array(tr_img_ids))
    print(f"✓ Train images saved {tr_img_feats.shape}")

tr_id2idx = {int(iid): idx for idx, iid in enumerate(tr_img_ids)}

# ── TRAIN TEXTS (chunked, resumable) ─────────────────────────────────────────
if os.path.exists(f"{CACHE_DIR}/tr_txt.npy"):
    print("✓ Train texts: loading from cache")
    tr_txt_feats    = np.load(f"{CACHE_DIR}/tr_txt.npy")
    tr_cap_img_ids  = np.load(f"{CACHE_DIR}/tr_capids.npy").tolist()
else:
    print("─── Encoding train texts (chunked, resumable) ───")
    tr_txt_feats, tr_cap_img_ids = encode_texts_chunked(
        text_enc,
        CaptionOnlyDS(COCO_ANN_TRAIN, tokenizer, MAX_SEQ_LEN),
        bs=1024, chunk_batches=100,
        save_dir=CACHE_DIR, prefix="tr_txt"
    )
    np.save(f"{CACHE_DIR}/tr_txt.npy",    tr_txt_feats)
    np.save(f"{CACHE_DIR}/tr_capids.npy", np.array(tr_cap_img_ids))
    print(f"✓ Train texts saved {tr_txt_feats.shape}")

tr_ann2img = np.array([tr_id2idx[int(i)] for i in tr_cap_img_ids])

# ── VAL IMAGES ────────────────────────────────────────────────────────────────
if os.path.exists(f"{CACHE_DIR}/va_img.npy"):
    print("✓ Val images: loading from cache")
    va_img_feats = np.load(f"{CACHE_DIR}/va_img.npy")
    va_img_ids   = np.load(f"{CACHE_DIR}/va_imgids.npy").tolist()
else:
    print("─── Encoding val images ───")
    va_img_feats, va_img_ids = encode_images(
        vis_enc, ImageOnlyDS(COCO_IMG_VAL, COCO_ANN_VAL, encode_tfm))
    np.save(f"{CACHE_DIR}/va_img.npy",    va_img_feats)
    np.save(f"{CACHE_DIR}/va_imgids.npy", np.array(va_img_ids))
    print(f"✓ Val images saved {va_img_feats.shape}")

va_id2idx = {int(iid): idx for idx, iid in enumerate(va_img_ids)}

# ── VAL TEXTS (chunked, resumable) ───────────────────────────────────────────
if os.path.exists(f"{CACHE_DIR}/va_txt.npy"):
    print("✓ Val texts: loading from cache")
    va_txt_feats   = np.load(f"{CACHE_DIR}/va_txt.npy")
    va_cap_img_ids = np.load(f"{CACHE_DIR}/va_capids.npy").tolist()
else:
    print("─── Encoding val texts (chunked, resumable) ───")
    va_txt_feats, va_cap_img_ids = encode_texts_chunked(
        text_enc,
        CaptionOnlyDS(COCO_ANN_VAL, tokenizer, MAX_SEQ_LEN),
        bs=1024, chunk_batches=50,
        save_dir=CACHE_DIR, prefix="va_txt"
    )
    np.save(f"{CACHE_DIR}/va_txt.npy",    va_txt_feats)
    np.save(f"{CACHE_DIR}/va_capids.npy", np.array(va_cap_img_ids))
    print(f"✓ Val texts saved {va_txt_feats.shape}")

va_ann2img = np.array([va_id2idx[int(i)] for i in va_cap_img_ids])

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'─'*50}")
print(f"Train: {tr_img_feats.shape[0]:,} imgs | {tr_txt_feats.shape[0]:,} caps")
print(f"Val  : {va_img_feats.shape[0]:,} imgs | {va_txt_feats.shape[0]:,} caps")
print(f"Cache dir contents:")
for f in sorted(os.listdir(CACHE_DIR)):
    mb = os.path.getsize(f"{CACHE_DIR}/{f}") / 1e6
    print(f"  {f:<40} {mb:>6.1f} MB")

In [ ]:
!pip install -q timm==0.4.12 einops fvcore causal-conv1d
!pip install -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.2cxx11abiTRUE-cp310-cp310-linux_x86_64.whl
!pip install -q pycocotools transformers

In [1]:
import os, sys, json, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE   = "cuda"
WORK_DIR = "/kaggle/working"
CACHE_DIR= f"{WORK_DIR}/feature_cache"
EMBED_DIM= 512
LR       = 5e-4
EPOCHS   = 30          # more epochs since projection only — each is ~5s
BATCH_SIZE_CACHED = 2048

N_GPUS = torch.cuda.device_count()
print(f"GPUs: {N_GPUS}  |  CUDA ok: {torch.cuda.is_available()}")

GPUs: 2  |  CUDA ok: True


In [3]:
# All features already on disk — encoders not needed at all for training
tr_img_feats = np.load(f"{CACHE_DIR}/tr_img.npy")
tr_txt_feats = np.load(f"{CACHE_DIR}/tr_txt.npy")
tr_cap_ids   = np.load(f"{CACHE_DIR}/tr_capids.npy").tolist()
tr_img_ids   = np.load(f"{CACHE_DIR}/tr_imgids.npy").tolist()

va_img_feats = np.load(f"{CACHE_DIR}/va_img.npy")
va_txt_feats = np.load(f"{CACHE_DIR}/va_txt.npy")
va_cap_ids   = np.load(f"{CACHE_DIR}/va_capids.npy").tolist()
va_img_ids   = np.load(f"{CACHE_DIR}/va_imgids.npy").tolist()

# Build ann2img index
tr_id2idx  = {int(iid): idx for idx, iid in enumerate(tr_img_ids)}
tr_ann2img = np.array([tr_id2idx[int(i)] for i in tr_cap_ids])

va_id2idx  = {int(iid): idx for idx, iid in enumerate(va_img_ids)}
va_ann2img = np.array([va_id2idx[int(i)] for i in va_cap_ids])

print(f"Train: {tr_img_feats.shape[0]:,} imgs | {tr_txt_feats.shape[0]:,} caps")
print(f"Val  : {va_img_feats.shape[0]:,} imgs | {va_txt_feats.shape[0]:,} caps")
print(f"img_dim={tr_img_feats.shape[1]}  txt_dim={tr_txt_feats.shape[1]}")
# Should print: img_dim=1024  txt_dim=768

Train: 118,287 imgs | 591,753 caps
Val  : 5,000 imgs | 25,014 caps
img_dim=1024  txt_dim=768


In [4]:
# ── Dataset: one image + one random caption per image per epoch ───────────────
class CachedPairDataset(Dataset):
    def __init__(self, img_feats, txt_feats, ann2img):
        self.img_f = torch.from_numpy(img_feats).float()
        self.txt_f = torch.from_numpy(txt_feats).float()
        # img_idx → list of caption indices
        self.img2caps = {}
        for cap_i, img_i in enumerate(ann2img):
            self.img2caps.setdefault(int(img_i), []).append(cap_i)
        self.img_indices = sorted(self.img2caps.keys())

    def __len__(self): return len(self.img_indices)

    def __getitem__(self, idx):
        img_i  = self.img_indices[idx]
        caps   = self.img2caps[img_i]
        cap_i  = caps[torch.randint(len(caps), (1,)).item()]
        return self.img_f[img_i], self.txt_f[cap_i]

# ── Model: projection heads only (encoders not needed at train time) ───────────
class ProjectionHead(nn.Module):
    def __init__(self, in_d, out_d):
        super().__init__()
        self.net  = nn.Sequential(nn.Linear(in_d, in_d), nn.GELU(), nn.Linear(in_d, out_d))
        self.norm = nn.LayerNorm(out_d)
    def forward(self, x): return self.norm(self.net(x))

class CachedCLIMP(nn.Module):
    def __init__(self, img_dim=1024, txt_dim=768, embed_dim=512):
        super().__init__()
        self.vis_proj   = ProjectionHead(img_dim,  embed_dim)
        self.text_proj  = ProjectionHead(txt_dim,  embed_dim)
        self.logit_scale = nn.Parameter(torch.ones([]) * math.log(1 / 0.07))

    def forward(self, img_f, txt_f):
        i = F.normalize(self.vis_proj(img_f),  dim=-1)
        t = F.normalize(self.text_proj(txt_f), dim=-1)
        s = self.logit_scale.exp().clamp(max=100)
        logits = s * i @ t.T
        return logits, logits.T

# ── Instantiate ───────────────────────────────────────────────────────────────
BATCH_SIZE_CACHED = 2048   # safe for T4×2; bump to 4096 if no OOM

train_ds     = CachedPairDataset(tr_img_feats, tr_txt_feats, tr_ann2img)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_CACHED,
                          shuffle=True, num_workers=4, pin_memory=True, drop_last=True)

proj_model = CachedCLIMP(img_dim=vis_enc.out_dim,
                          txt_dim=text_enc.out_dim,
                          embed_dim=EMBED_DIM).to(DEVICE)

# ── DataParallel across both T4s ──────────────────────────────────────────────
if N_GPUS > 1:
    proj_model = nn.DataParallel(proj_model)
    print(f"✓ DataParallel over {N_GPUS} GPUs")

trainable = sum(p.numel() for p in proj_model.parameters() if p.requires_grad)
print(f"Projection heads: {trainable:,} trainable params")
print(f"Train dataset   : {len(train_ds):,} image-caption pairs")
print(f"Batches/epoch   : {len(train_loader)} × batch {BATCH_SIZE_CACHED}")
# Expect: ~58 batches per epoch → full training in minutes, not hours

NameError: name 'vis_enc' is not defined

In [5]:
def infonce(logits_i2t, logits_t2i):
    B = logits_i2t.shape[0]
    labels = torch.arange(B, device=logits_i2t.device)
    return (F.cross_entropy(logits_i2t, labels) + F.cross_entropy(logits_t2i, labels)) / 2

# ── Unwrap helper (works whether DataParallel or not) ─────────────────────────
def unwrap(m): return m.module if hasattr(m, 'module') else m

# ── Checkpoint helpers ────────────────────────────────────────────────────────
CKPT_LATEST = f"{WORK_DIR}/climp_latest.pth"
CKPT_BEST   = f"{WORK_DIR}/climp_best.pth"

def save_ckpt(model, opt, sched, epoch, loss, best_loss):
    m = unwrap(model)
    state = dict(epoch=epoch, loss=loss, best_loss=best_loss,
                 vis_proj=m.vis_proj.state_dict(),
                 text_proj=m.text_proj.state_dict(),
                 logit_scale=m.logit_scale.data,
                 optimizer=opt.state_dict(),
                 scheduler=sched.state_dict())
    torch.save(state, CKPT_LATEST)
    if loss < best_loss:
        torch.save(state, CKPT_BEST)
        return True
    return False

def load_ckpt(model, opt, sched):
    if not os.path.exists(CKPT_LATEST):
        return 0, float('inf')
    state = torch.load(CKPT_LATEST, map_location='cpu')
    m = unwrap(model)
    m.vis_proj.load_state_dict(state['vis_proj'])
    m.text_proj.load_state_dict(state['text_proj'])
    m.logit_scale.data = state['logit_scale']
    opt.load_state_dict(state['optimizer'])
    sched.load_state_dict(state['scheduler'])
    print(f"✓ Resumed from epoch {state['epoch']+1}  loss={state['loss']:.4f}")
    return state['epoch'] + 1, state['best_loss']

# ── Optimizer + scheduler ─────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(proj_model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ── Resume if checkpoint exists (restart-safe) ────────────────────────────────
start_epoch, best_loss = load_ckpt(proj_model, optimizer, scheduler)

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"\nStarting from epoch {start_epoch+1}/{EPOCHS}")
print(f"Each epoch: {len(train_loader)} batches × {BATCH_SIZE_CACHED} = "
      f"{len(train_loader)*BATCH_SIZE_CACHED:,} pairs\n")

for epoch in range(start_epoch, EPOCHS):
    proj_model.train()
    t0 = time.time()
    total_loss = 0

    for img_f, txt_f in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        img_f = img_f.to(DEVICE)
        txt_f = txt_f.to(DEVICE)
        optimizer.zero_grad()
        logits_i2t, logits_t2i = proj_model(img_f, txt_f)
        loss = infonce(logits_i2t, logits_t2i)
        loss.backward()
        nn.utils.clip_grad_norm_(proj_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    elapsed  = time.time() - t0

    is_best = save_ckpt(proj_model, optimizer, scheduler, epoch, avg_loss, best_loss)
    if is_best: best_loss = avg_loss

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | loss={avg_loss:.4f} | "
          f"scale={unwrap(proj_model).logit_scale.exp().item():.1f} | "
          f"lr={scheduler.get_last_lr()[0]:.2e} | "
          f"time={elapsed:.1f}s {'← BEST' if is_best else ''}")

print(f"\n✓ Training complete. Best loss: {best_loss:.4f}")
print(f"  Latest : {CKPT_LATEST}")
print(f"  Best   : {CKPT_BEST}")

NameError: name 'proj_model' is not defined

In [ ]:
@torch.no_grad()
def evaluate(model, va_img_feats, va_txt_feats, va_ann2img, device, bs=2048):
    """Retrieval eval using cached val features — runs in seconds"""
    m = unwrap(model)
    m.eval()

    # Project cached features in batches to avoid OOM
    def project_batched(proj_fn, feats_np):
        out = []
        for i in range(0, len(feats_np), bs):
            chunk = torch.from_numpy(feats_np[i:i+bs]).float().to(device)
            out.append(F.normalize(proj_fn(chunk), dim=-1).cpu())
        return torch.cat(out)

    img_embs = project_batched(m.vis_proj,  va_img_feats)   # (N_imgs, 512)
    txt_embs = project_batched(m.text_proj, va_txt_feats)   # (N_txts, 512)

    # Full similarity matrix (5000 × 25000 for COCO val)
    sims = img_embs @ txt_embs.T    # (N_imgs, N_txts)

    # Build ground truth maps
    N_imgs = img_embs.shape[0]
    N_txts = txt_embs.shape[0]
    # img2caps[i] = list of caption indices belonging to image i
    img2caps = {}
    for cap_i, img_i in enumerate(va_ann2img):
        img2caps.setdefault(int(img_i), []).append(cap_i)

    def i2t_recall(k):
        """For each image, check if any GT caption in top-K"""
        hits = 0
        for img_i in range(N_imgs):
            gt_caps = set(img2caps.get(img_i, []))
            if not gt_caps: continue
            top_k = sims[img_i].topk(k).indices.tolist()
            if gt_caps & set(top_k): hits += 1
        return hits / N_imgs * 100

    def t2i_recall(k):
        """For each caption, check if GT image in top-K"""
        hits = 0
        sims_T = sims.T   # (N_txts, N_imgs)
        for cap_i in range(N_txts):
            gt_img = int(va_ann2img[cap_i])
            top_k  = sims_T[cap_i].topk(k).indices.tolist()
            if gt_img in top_k: hits += 1
        return hits / N_txts * 100

    print("\n── COCO Val Retrieval Results ───────────────────")
    print(f"{'Metric':<10} {'I→T':>8} {'T→I':>8}")
    print("─" * 30)
    results = {}
    for k in [1, 5, 10]:
        r_i2t = i2t_recall(k)
        r_t2i = t2i_recall(k)
        results[f'I2T_R@{k}'] = r_i2t
        results[f'T2I_R@{k}'] = r_t2i
        print(f"R@{k:<8} {r_i2t:>7.2f}% {r_t2i:>7.2f}%")
    print("─" * 30)
    return results

results = evaluate(proj_model, va_img_feats, va_txt_feats, va_ann2img, DEVICE)